In [7]:
# Enable IPython autoreload for iterative development
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Download All Annotated Article PDFs

Download all article PDFs linked to annotated Fuster dataset records to `data/pdfs/fuster` using robust fallback strategies:

- **Strategy 1**: OpenAlex PDF URL
- **Strategy 2**: Unpaywall API fallback
- **Strategy 3**: University EZproxy authentication (if configured)

This notebook processes all records in `dataset_article_mapping.csv` that have article DOIs, fetches OpenAlex metadata once per DOI, and downloads PDFs with error handling and retry logic.

**Features:**
- Single-pass OpenAlex API calls (cached)
- Batched downloads with polite rate limiting
- Comprehensive error tracking and manifest generation
- Support for EZproxy authentication for paywalled content

In [2]:
# Section 1: Imports and configuration
from pathlib import Path
import pandas as pd
import logging
import time

from llm_metadata.openalex import get_work_by_doi, extract_pdf_url
from llm_metadata.pdf_download import download_pdf_with_fallback
from llm_metadata.ezproxy import extract_cookies_from_browser

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger("download_all_fuster_pdfs")

# Output directory for all PDFs
OUTPUT_DIR = Path("data/pdfs/fuster")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Manifest path (to track all downloads)
MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Manifest path: {MANIFEST_PATH}")

Output directory: C:\Users\beav3503\dev\llm_metadata\data\pdfs\fuster
Manifest path: data\pdfs\fuster\manifest.csv


## Section 2: Load annotated dataset mappings

Load all records from the dataset-article mapping and filter for annotated entries (those with article DOIs).

In [3]:
# Load mapping CSV
mapping_path = Path("data/dataset_article_mapping.csv")
assert mapping_path.exists(), f"Mapping file not found: {mapping_path}"

df = pd.read_csv(mapping_path, dtype=str)
df.columns = [c.strip() for c in df.columns]

# Clean up whitespace
df['article_doi'] = df['article_doi'].fillna('').str.strip()
df['source'] = df['source'].fillna('').str.strip().str.lower()
df['dataset_doi'] = df['dataset_doi'].fillna('').str.strip()
df['title'] = df['title'].fillna('').str.strip()

# Filter: All rows with article DOI (annotated records)
annotated_df = df[(df['article_doi'] != '')].copy()
annotated_df.reset_index(drop=True, inplace=True)

print(f"Total records in mapping: {len(df)}")
print(f"Annotated records (with article DOI): {len(annotated_df)}")
print(f"\nBreakdown by source:")
print(annotated_df['source'].value_counts())

print(f"\nSample of annotated records:")
annotated_df[['dataset_doi', 'article_doi', 'source', 'title']].head(10)

Total records in mapping: 299
Annotated records (with article DOI): 75

Breakdown by source:
source
zenodo    38
dryad     37
Name: count, dtype: int64

Sample of annotated records:


,dataset_doi,article_doi,source,title
0,10.5061/dryad.0rxwdbs2c,10.5281/zenodo.5898699,zenodo,Exploration and diet specialization in eastern...
1,10.5061/dryad.121sk,10.1093/jhered/esx103,zenodo,Data from: Paternity analysis of wood turtles ...
2,10.5061/dryad.1771t,10.1371/journal.pone.0128238,dryad,Data from: Resampling method for applying dens...
3,10.5061/dryad.1cc4v,10.1371/journal.pone.0073695,zenodo,Data from: Impacts of human disturbance on lar...
4,10.5061/dryad.24q6q70,10.1002/ece3.4685,zenodo,Data from: Natal habitat preference induction ...
5,10.5061/dryad.24rj8,10.1639/0007-2745-119.1.008,dryad,"Data from: Aspicilia bicensis (Megasporaceae),..."
6,10.5061/dryad.2b8f1,10.1111/mec.14361,dryad,Data from: Do genetic drift and accumulation o...
7,10.5061/dryad.2bvq83brd,10.22541/au.161832268.87346989/v1,zenodo,Nonideal nest box selection by tree swallows b...
8,10.5061/dryad.2n5h6,10.1093/jhered/esw073,dryad,Data from: The genetic signature of range expa...
9,10.5061/dryad.36634,10.1002/ece3.3906,zenodo,Data from: Influence of northern limit range o...


## Section 3: Fetch OpenAlex works (single pass)

Fetch OpenAlex metadata **once** for each unique article DOI and cache the results. This avoids redundant API calls.

In [4]:
# Fetch OpenAlex works for each unique article DOI (single pass)
unique_dois = annotated_df['article_doi'].unique()
works_cache = {}

print(f"Fetching OpenAlex metadata for {len(unique_dois)} unique DOIs...\n")

for idx, doi in enumerate(unique_dois, 1):
    try:
        work = get_work_by_doi(doi)
        works_cache[doi] = work
        
        if work:
            oa_status = work.get('open_access', {}).get('oa_status', 'unknown')
            is_oa = work.get('open_access', {}).get('is_oa', False)
            logger.info(f"[{idx}/{len(unique_dois)}] ✓ {doi[:30]}... → {oa_status} {'(OA)' if is_oa else ''}")
        else:
            logger.warning(f"[{idx}/{len(unique_dois)}] ⚠ {doi[:30]}... → not found")
    
    except Exception as e:
        logger.error(f"[{idx}/{len(unique_dois)}] ✗ {doi[:30]}... → Error: {e}")
        works_cache[doi] = None
    
    # Be polite - sleep between requests (OpenAlex doesn't require this but good practice)
    if idx < len(unique_dois):
        time.sleep(0.5)

# Summary
found_count = sum(1 for w in works_cache.values() if w is not None)
print(f"\n{'='*60}")
print(f"OpenAlex fetch complete:")
print(f"  Found: {found_count}/{len(unique_dois)} ({100*found_count/len(unique_dois):.1f}%)")
print(f"  Missing: {len(unique_dois) - found_count}")
print(f"{'='*60}")

Fetching OpenAlex metadata for 75 unique DOIs...



INFO:download_all_fuster_pdfs:[1/75] ✓ 10.5281/zenodo.5898699... → green (OA)
INFO:download_all_fuster_pdfs:[2/75] ✓ 10.1093/jhered/esx103... → bronze (OA)
INFO:download_all_fuster_pdfs:[3/75] ✓ 10.1371/journal.pone.0128238... → gold (OA)
INFO:download_all_fuster_pdfs:[4/75] ✓ 10.1371/journal.pone.0073695... → gold (OA)
INFO:download_all_fuster_pdfs:[5/75] ✓ 10.1002/ece3.4685... → gold (OA)
INFO:download_all_fuster_pdfs:[6/75] ✓ 10.1639/0007-2745-119.1.008... → closed 
INFO:download_all_fuster_pdfs:[7/75] ✓ 10.1111/mec.14361... → closed 
INFO:download_all_fuster_pdfs:[8/75] ✓ 10.22541/au.161832268.87346989... → closed 
INFO:download_all_fuster_pdfs:[9/75] ✓ 10.1093/jhered/esw073... → bronze (OA)
INFO:download_all_fuster_pdfs:[10/75] ✓ 10.1002/ece3.3906... → gold (OA)
INFO:download_all_fuster_pdfs:[11/75] ✓ 10.1111/mec.12142... → closed 
INFO:download_all_fuster_pdfs:[12/75] ✓ 10.1111/ddi.12496... → bronze (OA)
INFO:download_all_fuster_pdfs:[13/75] ✓ 10.1002/ece3.1476... → gold (OA)
INF


OpenAlex fetch complete:
  Found: 71/75 (94.7%)
  Missing: 4


## Section 4: Download PDFs with fallback strategies

Download PDFs for all annotated records using `download_pdf_with_fallback()` which tries:
1. OpenAlex PDF URL (if available)
2. Unpaywall API fallback
3. EZproxy authentication (if configured)

In [8]:
# Try to extract cookies for EZProxy (optional - for paywalled content)
try:
    COOKIES = extract_cookies_from_browser(browser="firefox")
    if COOKIES:
        logger.info(f"✓ EZProxy cookies loaded ({len(COOKIES)} cookies)")
    else:
        logger.warning("⚠ No EZProxy cookies found - paywalled content may not be accessible")
except Exception as e:
    COOKIES = None
    logger.warning(f"⚠ EZProxy cookies failed: {e}")

results = []
total_records = len(annotated_df)

print(f"Downloading PDFs for {total_records} annotated records...\n")

for idx, row in annotated_df.iterrows():
    doi = row['article_doi']
    dataset_doi = row['dataset_doi']
    title = row.get('title', '')
    source = row.get('source', '')
    
    work = works_cache.get(doi)
    
    record = {
        'row_index': idx,
        'article_doi': doi,
        'dataset_doi': dataset_doi,
        'source': source,
        'title': title,
        'openalex_id': work.get('id') if work else None,
        'oa_status': work.get('open_access', {}).get('oa_status') if work else None,
        'is_oa': work.get('open_access', {}).get('is_oa', False) if work else False,
        'openalex_pdf_url': None,
        'downloaded_pdf_path': None,
        'status': 'pending',
        'error': None
    }
    
    # Skip if no OpenAlex work
    if not work:
        record['status'] = 'no_openalex_work'
        record['error'] = 'OpenAlex work not found'
        results.append(record)
        continue
    
    # Extract OpenAlex PDF URL (may be None)
    openalex_pdf_url = extract_pdf_url(work)
    record['openalex_pdf_url'] = openalex_pdf_url
    
    # Attempt download with fallback
    try:
        pdf_path = download_pdf_with_fallback(
            doi=doi,
            openalex_pdf_url=openalex_pdf_url,
            output_dir=OUTPUT_DIR,
            use_unpaywall=True,
            ezproxy_cookies=COOKIES
        )
        
        if pdf_path:
            record['status'] = 'downloaded'
            record['downloaded_pdf_path'] = str(pdf_path.relative_to(OUTPUT_DIR.parent))
            logger.info(f"[{idx+1}/{total_records}] ✓ {doi[:30]}... → {pdf_path.name}")
        else:
            record['status'] = 'failed'
            record['error'] = 'All download strategies failed'
            logger.warning(f"[{idx+1}/{total_records}] ✗ {doi[:30]}... → download failed")
    
    except Exception as e:
        record['status'] = 'error'
        record['error'] = str(e)
        logger.error(f"[{idx+1}/{total_records}] ✗ {doi[:30]}... → {e}")
    
    results.append(record)
    
    # Be polite between downloads
    time.sleep(0.3)

# Build results DataFrame
results_df = pd.DataFrame(results)
print(f"\n{'='*60}")
print(f"Download complete - {len(results_df)} records processed")
print(f"{'='*60}")

INFO:download_all_fuster_pdfs:✓ EZProxy cookies loaded (8 cookies)
INFO:llm_metadata.pdf_download:Strategy 2: Trying Unpaywall for 10.5281/zenodo.5898699


Extracting cookies from Firefox...
  ✓ Found 2 EZproxy cookie(s)
  ✓ Found 6 CAS cookie(s)
✓ Total: 8 cookie(s)



INFO:llm_metadata.pdf_download:Strategy 3: Trying EZproxy for 10.5281/zenodo.5898699
INFO:llm_metadata.pdf_download:Trying EZproxy ezproxy_login for 10.5281/zenodo.5898699
INFO:llm_metadata.pdf_download:Strategy 4: Trying Sci-Hub for 10.5281/zenodo.5898699
INFO:Sci-Hub:Failed to fetch pdf with identifier 10.5281/zenodo.5898699 (resolved url None) due to request exception.
ERROR:llm_metadata.pdf_download:All download strategies failed for 10.5281/zenodo.5898699
INFO:llm_metadata.pdf_download:PDF already exists: data\pdfs\fuster\10.1093_jhered_esx103.pdf
INFO:download_all_fuster_pdfs:[2/75] ✓ 10.1093/jhered/esx103... → 10.1093_jhered_esx103.pdf
INFO:llm_metadata.pdf_download:PDF already exists: data\pdfs\fuster\10.1371_journal.pone.0128238.pdf
INFO:download_all_fuster_pdfs:[3/75] ✓ 10.1371/journal.pone.0128238... → 10.1371_journal.pone.0128238.pdf
INFO:llm_metadata.pdf_download:PDF already exists: data\pdfs\fuster\10.1371_journal.pone.0073695.pdf
INFO:download_all_fuster_pdfs:[4/75] ✓ 10


Download complete - 75 records processed


## Section 5: Build manifest and summary metrics

Save results to CSV manifest and display comprehensive statistics.

In [9]:
# # Save manifest
# results_df.to_csv(MANIFEST_PATH, index=False)

# # Summary statistics
# summary = results_df['status'].value_counts()

# print("\n" + "="*70)
# print("DOWNLOAD SUMMARY")
# print("="*70)

# for status, count in summary.items():
#     pct = 100 * count / len(results_df)
#     print(f"{status:20s}: {count:4d} ({pct:5.1f}%)")

# print(f"\n✓ Manifest saved to: {MANIFEST_PATH}")
# print(f"✓ PDFs saved to: {OUTPUT_DIR.resolve()}")

# # Statistics by OA status
# print(f"\n{'='*70}")
# print("OA STATUS BREAKDOWN (for records with OpenAlex data)")
# print("="*70)

# oa_df = results_df[results_df['oa_status'].notna()]
# if len(oa_df) > 0:
#     oa_stats = oa_df['oa_status'].value_counts()
#     for status, count in oa_stats.items():
#         pct = 100 * count / len(oa_df)
#         print(f"{status:20s}: {count:4d} ({pct:5.1f}%)")
# else:
#     print("No OA status information available")

# # Show successful downloads
# downloaded = results_df[results_df['status'] == 'downloaded']
# if len(downloaded) > 0:
#     print(f"\n{'='*70}")
#     print(f"✓ SUCCESSFULLY DOWNLOADED: {len(downloaded)} PDFs")
#     print("="*70)
#     for _, row in downloaded.iterrows():
#         print(f"  • {row['article_doi'][:35]:35s} [{row['oa_status']}]")
#         if row['title']:
#             title_preview = row['title'][:60]
#             print(f"    └─ {title_preview}...")
# else:
#     print(f"\n✗ No PDFs were successfully downloaded")

# # Show failures with details
# failed = results_df[results_df['status'].isin(['failed', 'error', 'no_openalex_work'])]
# if len(failed) > 0:
#     print(f"\n{'='*70}")
#     print(f"✗ FAILED/SKIPPED: {len(failed)} records")
#     print("="*70)
    
#     for status_type in ['no_openalex_work', 'failed', 'error']:
#         subset = failed[failed['status'] == status_type]
#         if len(subset) > 0:
#             print(f"\n  {status_type.upper()} ({len(subset)}):")
#             for _, row in subset.head(10).iterrows():
#                 print(f"    • {row['article_doi'][:35]:35s} [{row['oa_status']}]")
#                 if row['error']:
#                     print(f"      Error: {row['error']}")
#             if len(subset) > 10:
#                 print(f"    ... and {len(subset) - 10} more")


DOWNLOAD SUMMARY
downloaded          :   70 ( 93.3%)
no_openalex_work    :    4 (  5.3%)
failed              :    1 (  1.3%)

✓ Manifest saved to: data\pdfs\fuster\manifest.csv
✓ PDFs saved to: C:\Users\beav3503\dev\llm_metadata\data\pdfs\fuster

OA STATUS BREAKDOWN (for records with OpenAlex data)
gold                :   25 ( 35.2%)
bronze              :   22 ( 31.0%)
closed              :   20 ( 28.2%)
green               :    4 (  5.6%)

✓ SUCCESSFULLY DOWNLOADED: 70 PDFs
  • 10.1093/jhered/esx103               [bronze]
    └─ Data from: Paternity analysis of wood turtles (Glyptemys ins...
  • 10.1371/journal.pone.0128238        [gold]
    └─ Data from: Resampling method for applying density-dependent ...
  • 10.1371/journal.pone.0073695        [gold]
    └─ Data from: Impacts of human disturbance on large prey specie...
  • 10.1002/ece3.4685                   [gold]
    └─ Data from: Natal habitat preference induction in large mamma...
  • 10.1639/0007-2745-119.1.008         [clos